# Notebook 3: Unsupervised Modelling
## CS336 - Artificial Intelligence and Machine Learning (AIML)
### Assignment: Smart Energy Consumption Advisory Agent

**Purpose:** This notebook applies unsupervised learning techniques to the engineered features:
1. **K-Means Clustering** — groups time interva...

In [ ]:
# ─── Imports ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sns.set_theme(style='whitegrid')
np.random.seed(42)  # Reproducibility
print('Imports successful.')

## 1. Load Engineered Data

In [ ]:
# ─── Load engineered feature dataset from Notebook 2 ─────────────────────────
df = pd.read_csv('data_engineered.csv', parse_dates=['timestamp'], index_col='timestamp')

# Select numeric features relevant to clustering (exclude label-like columns)
feature_cols = ['global_active_power', 'sub_metering_1', 'sub_metering_2',
                'sub_metering_3', 'rolling_mean_1h', 'rolling_std_1h',
                'lag_1', 'hour_sin', 'hour_cos']

X = df[feature_cols].fillna(0).values  # Convert to NumPy array for sklearn

# Standardise: zero mean, unit variance — critical for K-Means and DBSCAN
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Feature matrix shape: {X_scaled.shape}')

## 2. K-Means Clustering

### 2a.

In [ ]:
# ─── Elbow method on a stratified 10 % sample (faster convergence) ───────────
# Using a sample avoids long runtimes while still revealing the elbow faithfully
sample_size = int(0.1 * len(X_scaled))
idx_sample = np.random.choice(len(X_scaled), sample_size, replace=False)
X_sample = X_scaled[idx_sample]

k_range = range(2, 11)  # Evaluate k from 2 to 10
inertia_vals = []
sil_vals = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_sample)
    inertia_vals.append(km.inertia_)                          # WCSS
    sil_vals.append(silhouette_score(X_sample, labels))       # Silhouette

# ── Plot elbow curve and silhouette scores side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_range), inertia_vals, marker='o', color='steelblue')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method — Inertia vs. k')

axes[1].plot(list(k_range), sil_vals, marker='s', color='darkorange')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs. k (higher = better)')

plt.suptitle('K-Means: Optimal k Selection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2b. Fit Final K-Means Model (k = 4)

In [ ]:
# ─── Train K-Means with chosen k ──────────────────────────────────────────────
OPTIMAL_K = 4  # Selected from elbow / silhouette analysis above

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=20)
df['cluster'] = kmeans.fit_predict(X_scaled)  # Assign each row to a cluster

print(f'K-Means converged in {kmeans.n_iter_} iterations.')
print('Cluster sizes:')
print(df['cluster'].value_counts().sort_index())

### 2c. Cluster Profile Interpretation

In [ ]:
# ─── Cluster centroids (back in original scale) ───────────────────────────────
# Inverse-transform centroids so they are interpretable in physical units (kW, Wh)
centroids_orig = scaler.inverse_transform(kmeans.cluster_centers_)
centroid_df = pd.DataFrame(centroids_orig, columns=feature_cols)
centroid_df.index.name = 'Cluster'

print('Cluster centroids (original units):')
centroid_df[['global_active_power', 'sub_metering_1',
             'sub_metering_2', 'sub_metering_3']].round(3)

## 3. Anomaly Detection — Isolation Forest

**Isolation Forest** isolates observations by randomly partitioning features.
Anomalous points (unusual consumption spikes or drops) require fewer splits and are flagged.

In [ ]:
# ─── Isolation Forest: fit and predict anomaly labels ───────────────────────
# contamination = expected proportion of anomalies in the data
iso = IsolationForest(n_estimators=200,
                      contamination=0.02,  # ~2 % anomaly rate expected
                      random_state=42)
df['anomaly'] = iso.fit_predict(X_scaled)  # -1 = anomaly, +1 = normal

# Convert to a binary flag for readability
df['is_anomaly'] = (df['anomaly'] == -1).astype(int)

n_anomalies = df['is_anomaly'].sum()
print(f'Anomalies detected: {n_anomalies} ({n_anomalies/len(df)*100:.2f}% of dataset)')

In [ ]:
# ─── Visualise anomalies over time ───────────────────────────────────────────
# Plot a 7-day window to see how anomalies appear within the time series
window_df = df['2023-06-01':'2023-06-07']  # One representative week

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(window_df.index, window_df['global_active_power'],
        color='steelblue', linewidth=0.8, label='Power (kW)')

# Overlay anomaly points in red
anomaly_pts = window_df[window_df['is_anomaly'] == 1]
ax.scatter(anomaly_pts.index, anomaly_pts['global_active_power'],
           color='red', zorder=5, s=40, label='Anomaly')

ax.set_title('Global Active Power with Anomalies Highlighted (June 1–7)', fontsize=13)
ax.set_xlabel('Timestamp')
ax.set_ylabel('Active Power (kW)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. DBSCAN Clustering

**DBSCAN** (Density-Based Spatial Clustering of Applications with Noise) finds clusters  
of arbitrary shape and marks low-density points as noise — complementing K-Means.

In [ ]:
# ─── DBSCAN on PCA-reduced 2-D data (for interpretable visualisation) ────────
# Reducing to 2 PCs before DBSCAN avoids the curse of dimensionality
pca2 = PCA(n_components=2, random_state=42)
X2   = pca2.fit_transform(X_scaled)  # Project to 2-D

# eps = neighbourhood radius; min_samples = core point threshold
dbscan = DBSCAN(eps=0.5, min_samples=30)
db_labels = dbscan.fit_predict(X2)   # -1 = noise, 0,1,2,… = cluster IDs

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise       = (db_labels == -1).sum()
print(f'DBSCAN clusters found: {n_clusters_db} | Noise points: {n_noise}')

# ── Scatter plot of DBSCAN results in 2-D PCA space
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X2[:, 0], X2[:, 1],
                      c=db_labels, cmap='tab10', alpha=0.4, s=3)
plt.colorbar(scatter, label='DBSCAN Cluster (-1 = noise)')
plt.title('DBSCAN Clusters in 2-D PCA Space')
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Save modelling results (cluster labels + anomaly flags) ─────────────────
df.to_csv('data_modelled.csv')
print('Saved: data_modelled.csv')
print('Columns added: cluster, anomaly, is_anomaly')